# 02 · Three short calls: hedge frequency and diversification

**Objective:** extend notebook 01's accounting to short JPM, PNC and GS calls. Compare initial-only, daily, two-day and five-session hedging under independent and correlated stock-return shocks.

Each policy is evaluated on the **same paths within a dependence scenario**. Portfolio P&L is the sum of the three positions' P&Ls for each joint outcome. Portfolio tail risk is calculated from those aggregate outcomes.

The baseline uses **5,000 paths per scenario and zero stock trading costs**, isolating residual hedge risk and dependence. Notebook 03 studies costs and volatility mismatch. Initial-only means an initial delta hedge held until expiry; it is not an unhedged call position.

This completed notebook runs independently from a fresh kernel. Model inputs load from the same validated study file as notebook 00; the policy controls remain explicit. All inputs are illustrative. Paths use Q-GBM with zero dividends; the book has synthetic European calls with share-equivalent units and no exchange contract multiplier.

**Shared study inputs:** contract and scenario assumptions now load from `config/illustrative_study.json` through `load_named_inputs`. `Book` remains the validated model container. Change that file once to update the baseline across notebooks 00–03; policy/experiment controls stay explicit in each notebook. 


| Step | Purpose | Main functions/tools |
|---|---|---|
| 1–2 | Inspect named inputs and opening exposures | Dictionaries, `Book`, `initial_exposures` |
| 3 | Generate independent and correlated paths | `simulate_paths` |
| 4 | Evaluate all four hedge policies on each path set | `hedge`, `summarize` |
| 5 | Verify the aggregate cash account | Grouped audit ledger and NumPy assertions |
| 6 | Compare risk, turnover and outcome distributions | Tables, bar charts, histograms |
| 7 | Explain portfolio variance and P&L dependence | Covariance and correlation matrices |
| 8 | Attribute portfolio-tail losses to each name | Common portfolio-tail observations |
| 9–10 | Interpret the comparisons and record the run | Calculated ratios, `provenance` |

Cash attribution across names sums to one shared account because borrowing and lending use the same constant rate and there are no margin constraints. The code below checks the resulting accounting identities for every scenario/policy combination.

## 1 · Connect to the project

Find the project root, then import the same pricing and accounting helpers used in notebooks 00 and 01. Either the project root or notebook folder can be the kernel's starting directory.

In [ ]:
from hedging_harness.notebook_views import run
_ = run('02_three_name_portfolio')


## 2 · Define the named portfolio and policies

Each security owns its contract and pricing inputs. The scenario owns the path volatilities and named correlation pairs. `notional / spot` gives each short-call quantity, and sessions divided by sessions per year gives maturity in years.

`POLICIES` maps policy names to rebalance intervals. All policies open an initial delta hedge and liquidate stock at expiry. `COST_BPS = 0.0` is the baseline here; if changed, the reported net P&L automatically includes those costs.

`AUDIT_PATH` retains one full path's per-name accounting for each of the eight cases. Every simulated outcome contributes to the distribution statistics.

## 3 · Generate independent and correlated outcomes

The two scenarios keep the same spots, strikes, rates, path volatilities, pricing volatilities and random seed. Only the stock-shock correlation changes:

- **independent:** identity correlation matrix.
- **illustrative_correlated:** the configured named-pair matrix, matching notebook 00.

Each scenario uses one path set for every hedge policy. Across dependence scenarios, the same underlying normal draws are transformed differently, so the resulting price paths differ. Finite-sample marginal results may change; standalone option fair values do not change when their own pricing inputs stay fixed.

The empirical correlation check uses one-session **log returns**, pooling independent path/session increments. Price-level correlation would be a different diagnostic.

## 4 · Apply all four policies to each path set

For each scenario, `hedge(...)` runs every policy on the same realized stock histories. Its output contains one terminal P&L per outcome and per name. Sum across names **within each outcome** to obtain the joint portfolio result, then compute portfolio statistics.

`summarize(...)` reports means, Monte Carlo mean SEs, P&L standard deviation, RMSE, fifth-percentile P&L, expected shortfall, costs and turnover. ES95 is minus the mean of the worst 5% of outcomes, rounded up to a whole count. It can be negative when even that tail is profitable.

With matched pricing/path volatilities and zero costs, predictable stock hedging has zero expected discounted gain under Q. The comparison concerns residual risk and trading activity; sample mean differences are not evidence of alpha.

## 5 · Verify the shared-account aggregation

Each name has its own attributed cash balance, but all use the same linear financing rate. Summing those balances therefore gives the shared cash account. Stock holdings remain distinct because JPM shares and PNC shares are different assets.

For the audited outcome in **every** scenario/policy case, sum the cash flows across names at each date. The aggregate account must satisfy

$$\Delta C = \text{premium}+\text{interest}+\text{stock cash flow}-\text{cost}-\text{payoff paid}.$$

Marked portfolio value is cash plus stock value minus option liabilities. At expiry, all shares and liabilities must be zero, and final shared cash must equal the sum of the three terminal P&Ls. The tables below display the correlated five-session case as an example; assertions cover all eight cases.

## 6 · Compare hedge frequency, risk and turnover

The risk table compares each policy's P&L standard deviation with both the daily hedge and the initial-only hedge. A ratio above one in `std_relative_to_daily` means more residual variability than daily hedging in that scenario. `std_reduction_vs_initial_pct` measures the percentage reduction relative to initial-only.

The charts compare P&L standard deviation, portfolio ES95 and total dollar turnover. Lower variability can require more trading. With the zero-cost baseline, turnover is an activity measure; costs are not yet used to select a policy.

P&L standard deviation describes simulated economic variability. Mean SE measures uncertainty in the estimated mean and should not be substituted for that risk measure.

### Compare the aggregate distributions

The next figure overlays daily and five-session portfolio P&L distributions in each dependence scenario. The panels use a common P&L axis and common histogram bins, making their spread and tails comparable.

The name-level table underneath shows daily results in the correlated scenario. The portfolio row is computed from joint summed P&L, not from a sum of name-level standard deviations or tail statistics.

## 7 · Explain portfolio variance and P&L correlations

For position P&Ls $X_i$,

$$\operatorname{Var}\!\left(\sum_i X_i\right)
=\sum_i\operatorname{Var}(X_i)+2\sum_{i<j}\operatorname{Cov}(X_i,X_j).$$

`own_variances` is the covariance matrix's trace. `cross_covariances` is the sum of its off-diagonal entries, which already includes both occurrences of every pair. Their sum must match the variance of the pathwise aggregate portfolio P&L.

`std_diversification_ratio` compares portfolio standard deviation with the sum of standalone standard deviations. Values below one indicate diversification under that measure. This is an accounting decomposition of simulated outcomes, rather than a causal allocation of business risks.

Stock-return correlation and hedge-error P&L correlation are different quantities. Discrete hedge errors depend nonlinearly on the paths, so their correlations need not match the configured shock correlations. Even independent simulations have small sample covariance rather than exactly zero covariance.

## 8 · Attribute the portfolio's worst outcomes

Select the worst 5% of **portfolio** outcomes, then calculate each name's average loss on those same observations. These contributions sum to the portfolio ES95. They answer which name contributes most when the combined book experiences its worst outcomes.

A standalone name's ES95 selects that name's own worst observations, which may occur on different paths. Adding those standalone ES95 values therefore does not calculate portfolio ES95. The comparison table reports their sum alongside the actual portfolio statistic to make the distinction explicit.

Contributions can be negative if a name offsets losses in the selected portfolio tail. Percentage contributions can exceed 100% when another name offsets losses. These are descriptive sample allocations and do not quantify tail-estimation uncertainty.

## 9 · Read the actual comparisons

The calculated summary below answers the original study prompts: how much variability a five-session hedge retains relative to daily hedging, how diversification changes with dependence, and which name contributes the most to the portfolio tail at equal notional.

These comparisons use the displayed simulated sample. They do not establish an optimal policy or a historical expected return. A comparison of means needs Monte Carlo uncertainty; a comparison of tails additionally needs its own uncertainty analysis. Dependence scenarios share base normal draws but are different price paths, so a same-path policy-comparison function should not be applied to them as if the stock histories were identical.

## 10 · Record the full experiment

The provenance record captures the book, path volatilities, both dependence matrices, seed, path count, hedge policies, trading-cost assumption and audit path. It also records the named inputs, runtime versions, source-file hashes and a hash of this notebook's code.

The base `scenario` field identifies the configured correlated benchmark; `dependence_scenarios` records every matrix actually used. Pair-keyed correlations are represented as labelled list entries for JSON serialization. The notebook displays the record without writing a separate export.

## What this notebook establishes

The one-option ledger extends to a three-name book with consistent shared-account aggregation. Each dependence scenario evaluates four hedge frequencies on common paths. The resulting tables and figures show residual risk, trading activity, variance/covariance effects and contributions to the joint portfolio tail.

The independent and correlated scenarios retain the same marginal pricing inputs. Their joint outcomes and portfolio risks differ, while finite-sample estimates can also move. A hedge that removes initial delta still leaves gamma exposure, discrete-hedging error and implementation activity.

Next: **03_cost_and_volatility_sensitivity.ipynb** studies trading costs and hedge-volatility assumptions. It loads the same shared model assumptions and declares its cost/volatility experiments explicitly. Dividends, option spreads, margin and exercise/assignment remain outside this illustrative model.